# 04 — Analysis: plots and results

Load the Phase 6 sweep outputs (`results/benchmarks/benchmark_*.csv`) and plot
throughput, latency (TTFT/TPOT), MFU, and power across backends and batch sizes.
Figures are written to `results/plots/`.

In [ ]:
# ── Environment check ──────────────────────────────────────────
import sys
sys.path.insert(0, "/content/llm-inference-optimizer")  # Colab path
sys.path.insert(0, "..")                                  # local path fallback

from src.utils.env import get_env_info
info = get_env_info()
print(f"Device   : {info['device']}")
print(f"GPU      : {info['gpu_name']}")
print(f"CUDA     : {info['cuda_version']}")
print(f"PyTorch  : {info['torch_version']}")
print(f"Colab    : {info['is_colab']}")

## 1. Load all benchmark results

In [ ]:
import glob
import pandas as pd

files = sorted(glob.glob("results/benchmarks/benchmark_*.csv"))
assert files, "No benchmark CSVs found — run 03_benchmark first."
df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
print(f"Loaded {len(df)} rows from {len(files)} file(s); backends: {sorted(df.backend.unique())}")
df.head()

## 2. Summary table (mean per backend)

In [ ]:
summary = (df.groupby('backend')[['throughput_tps', 'ttft_ms', 'tpot_ms', 'mfu_percent', 'power_watts']]
            .mean().round(2))
summary

## 3. Plots

One line per backend vs batch size (at the largest common seq_len). Saved to
`results/plots/`.

In [ ]:
import os
import matplotlib.pyplot as plt

os.makedirs("results/plots", exist_ok=True)
seq = sorted(df.seq_len.unique())[-1]          # largest seq_len present
sub = df[df.seq_len == seq]

metrics = [
    ("throughput_tps", "Throughput (tok/s)"),
    ("ttft_ms", "TTFT (ms)"),
    ("tpot_ms", "TPOT (ms)"),
    ("mfu_percent", "MFU (%)"),
    ("power_watts", "Power (W)"),
]
fig, axes = plt.subplots(1, len(metrics), figsize=(5 * len(metrics), 4))
for ax, (col, label) in zip(axes, metrics):
    for backend, g in sub.groupby("backend"):
        g = g.sort_values("batch_size")
        ax.plot(g.batch_size, g[col], marker="o", label=backend)
    ax.set_xlabel("batch size"); ax.set_ylabel(label); ax.set_title(label)
    ax.legend(); ax.grid(True, alpha=0.3)
fig.suptitle(f"Backend comparison @ seq_len={seq}")
fig.tight_layout()
fig.savefig("results/plots/backend_comparison.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved results/plots/backend_comparison.png")

## 4. Speedup vs eager baseline

The headline metrics: throughput speedup and latency reduction of each backend
relative to the `eager` baseline (matched on batch_size + seq_len).

In [ ]:
if 'eager' in df.backend.unique() and df.backend.nunique() > 1:
    base = df[df.backend == 'eager'].set_index(['batch_size', 'seq_len'])
    for backend in sorted(b for b in df.backend.unique() if b != 'eager'):
        cur = df[df.backend == backend].set_index(['batch_size', 'seq_len'])
        common = base.index.intersection(cur.index)
        tps_speedup = (cur.loc[common, 'throughput_tps'] / base.loc[common, 'throughput_tps']).mean()
        lat_red = (1 - cur.loc[common, 'tpot_ms'] / base.loc[common, 'tpot_ms']).mean()
        print(f"{backend:6s} vs eager:  throughput {tps_speedup:.2f}x  |  TPOT latency -{lat_red*100:.1f}%")
else:
    print("Need eager + >=1 other backend (run 03_benchmark for each) to compute speedups.")

In [ ]:
# ── Save results to GitHub ─────────────────────────────────────
import subprocess
NOTEBOOK_NAME = "04_analysis"
subprocess.run(["git", "add", "results/", "notebooks/"], check=True)
subprocess.run(["git", "commit", "-m", f"results: {NOTEBOOK_NAME} run"], check=True)
subprocess.run(["git", "push"], check=True)
print("Results pushed to GitHub.")